# Visualize Baseline Model

Use this notebook to inspect the original base model before any adapters are applied. Set `checkpoint_dir` below, and the notebook will use it only to resolve the base model name before loading and summarizing the baseline model itself.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from IPython.display import Markdown, display
from torchinfo import summary
from transformers import AutoModelForCausalLM, AutoTokenizer


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd()

checkpoint_dir = "sft-arithmetic-lora-demo/checkpoint-19"

device_map = "auto"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

summary_depth = 4
summary_verbose = 1
summary_col_names = ["input_size", "output_size", "num_params", "trainable"]
summary_row_settings = ["var_names", "depth"]

print(f"Repo root: {repo_root}")
print(f"Selected checkpoint_dir: {checkpoint_dir}")
print(f"torch_dtype: {torch_dtype}")
print(f"device_map: {device_map}")


Repo root: /Users/jim/Desktop/genai/rft-learning
Selected checkpoint_dir: sft-arithmetic-lora-demo/checkpoint-19
torch_dtype: torch.float32
device_map: auto


In [3]:
def discover_checkpoint_candidates(root: Path) -> list[Path]:
    return sorted(
        path for path in root.glob("*-demo/checkpoint-*")
        if (path / "adapter_config.json").exists()
    )


def load_json(path: Path) -> dict[str, Any]:
    with path.open() as fh:
        return json.load(fh)


def format_int(value: int | float | None) -> str:
    if value is None:
        return "-"
    return f"{int(value):,}"


def format_percent(part: int, whole: int) -> str:
    if whole == 0:
        return "0.00%"
    return f"{(part / whole) * 100:.2f}%"


def normalize_path(path_str: str, root: Path) -> Path:
    path = Path(path_str)
    return path if path.is_absolute() else (root / path)


def parameter_stats(model: torch.nn.Module) -> dict[str, int]:
    total = 0
    trainable = 0

    for _, param in model.named_parameters():
        count = param.numel()
        total += count
        if param.requires_grad:
            trainable += count

    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
    }


def shared_parameter_count(model: torch.nn.Module) -> int:
    unique_total = sum(param.numel() for _, param in model.named_parameters())
    all_named_total = sum(param.numel() for _, param in model.named_parameters(remove_duplicate=False))
    return all_named_total - unique_total


def dtype_breakdown(model: torch.nn.Module) -> pd.DataFrame:
    counts: dict[str, int] = {}
    for _, param in model.named_parameters():
        key = str(param.dtype)
        counts[key] = counts.get(key, 0) + param.numel()

    rows = [
        {"dtype": dtype_name, "parameter_count": format_int(count)}
        for dtype_name, count in sorted(counts.items())
    ]
    return pd.DataFrame(rows)


checkpoint_candidates = discover_checkpoint_candidates(repo_root)

display(Markdown("## Checkpoint Candidates"))
display(pd.DataFrame({"checkpoint_dir": [str(path.relative_to(repo_root)) for path in checkpoint_candidates]}))


## Checkpoint Candidates

,checkpoint_dir
0,grpo-arithmetic-lora-demo/checkpoint-72
1,grpo-arithmetic-lora-early-stopping-demo/check...
2,grpo-arithmetic-lora-early-stopping-demo/check...
3,sft-arithmetic-lora-demo/checkpoint-19


In [4]:
checkpoint_path = normalize_path(checkpoint_dir, repo_root)

if not checkpoint_path.exists():
    candidate_list = "\n".join(f"- {path.relative_to(repo_root)}" for path in checkpoint_candidates)
    raise FileNotFoundError(
        "Checkpoint directory was not found. Set `checkpoint_dir` to one of:\n" + candidate_list
    )

adapter_config_path = checkpoint_path / "adapter_config.json"
if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"Expected adapter_config.json at {adapter_config_path} so the notebook can locate the original base model."
    )

adapter_config = load_json(adapter_config_path)
base_model_name = adapter_config.get("base_model_name_or_path")
if not base_model_name:
    raise ValueError("adapter_config.json does not contain base_model_name_or_path")

display(Markdown("## Resolved Base Model"))
display(pd.DataFrame([{
    "checkpoint_source": str(checkpoint_path.relative_to(repo_root)),
    "base_model_name_or_path": base_model_name,
}]))


## Resolved Base Model

,checkpoint_source,base_model_name_or_path
0,sft-arithmetic-lora-demo/checkpoint-19,Qwen/Qwen2.5-0.5B-Instruct


In [5]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs: dict[str, Any] = {"dtype": torch_dtype}
if device_map is not None:
    load_kwargs["device_map"] = device_map

model = AutoModelForCausalLM.from_pretrained(base_model_name, **load_kwargs)
model.eval()

first_parameter = next(model.parameters())
model_device = first_parameter.device

example_batch = tokenizer(
    "What is 9 + 3?",
    return_tensors="pt",
    padding=False,
    truncation=True,
)
example_batch = {key: value.to(model_device) for key, value in example_batch.items()}

display(Markdown("## Loaded Baseline Model"))
display(pd.DataFrame([{
    "model_class": type(model).__name__,
    "model_device": str(model_device),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_name_or_path": getattr(tokenizer, 'name_or_path', '-'),
    "pad_token": tokenizer.pad_token or '-',
    "eos_token": tokenizer.eos_token or '-',
}]))


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1384.93it/s]


## Loaded Baseline Model

,model_class,model_device,tokenizer_class,tokenizer_name_or_path,pad_token,eos_token
0,Qwen2ForCausalLM,mps:0,Qwen2Tokenizer,Qwen/Qwen2.5-0.5B-Instruct,<|endoftext|>,<|im_end|>


In [6]:
display(Markdown("## torchinfo Summary"))
display(Markdown("`torchinfo` may count shared or tied weights more than once. The report below uses unique parameter totals from `named_parameters()`."))

with torch.no_grad():
    model_summary = summary(
        model,
        input_data=example_batch,
        depth=summary_depth,
        col_names=summary_col_names,
        row_settings=summary_row_settings,
        verbose=summary_verbose,
        device=str(model_device),
    )

model_summary


## torchinfo Summary

`torchinfo` may count shared or tied weights more than once. The report below uses unique parameter totals from `named_parameters()`.

Layer (type (var_name):depth-idx)                                      Input Shape               Output Shape              Param #                   Trainable
Qwen2ForCausalLM (Qwen2ForCausalLM)                                    --                        --                        --                        True
├─Qwen2Model (model): 1-1                                              --                        --                        --                        True
│    └─Embedding (embed_tokens): 2-1                                   [1, 8]                    [1, 8, 896]               136,134,656               True
│    └─Qwen2RotaryEmbedding (rotary_emb): 2-2                          [1, 8, 896]               [1, 8, 64]                --                        --
│    └─ModuleList (layers): 2-3                                        --                        --                        --                        True
│    │    └─Qwen2DecoderLayer (0): 3-1                                 [1

Layer (type (var_name):depth-idx)                                      Input Shape               Output Shape              Param #                   Trainable
Qwen2ForCausalLM (Qwen2ForCausalLM)                                    --                        --                        --                        True
├─Qwen2Model (model): 1-1                                              --                        --                        --                        True
│    └─Embedding (embed_tokens): 2-1                                   [1, 8]                    [1, 8, 896]               136,134,656               True
│    └─Qwen2RotaryEmbedding (rotary_emb): 2-2                          [1, 8, 896]               [1, 8, 64]                --                        --
│    └─ModuleList (layers): 2-3                                        --                        --                        --                        True
│    │    └─Qwen2DecoderLayer (0): 3-1                                 [1

In [7]:
stats = parameter_stats(model)
shared_params = shared_parameter_count(model)

model_report = pd.DataFrame(
    [
        {"metric": "Selected checkpoint path", "value": str(checkpoint_path.relative_to(repo_root))},
        {"metric": "Resolved base model", "value": base_model_name},
        {"metric": "Loaded model class", "value": type(model).__name__},
        {"metric": "Total parameters", "value": format_int(stats['total'])},
        {"metric": "Trainable parameters", "value": f"{format_int(stats['trainable'])} ({format_percent(stats['trainable'], stats['total'])})"},
        {"metric": "Frozen parameters", "value": f"{format_int(stats['frozen'])} ({format_percent(stats['frozen'], stats['total'])})"},
        {"metric": "Shared/tied parameters counted separately by torchinfo", "value": format_int(shared_params)},
    ]
)

tokenizer_report = pd.DataFrame(
    [
        {"field": "tokenizer_class", "value": type(tokenizer).__name__},
        {"field": "tokenizer_name_or_path", "value": getattr(tokenizer, 'name_or_path', '-')},
        {"field": "pad_token", "value": tokenizer.pad_token or '-'},
        {"field": "eos_token", "value": tokenizer.eos_token or '-'},
        {"field": "vocab_size", "value": getattr(tokenizer, 'vocab_size', '-')},
        {"field": "model_max_length", "value": getattr(tokenizer, 'model_max_length', '-')},
    ]
)

display(Markdown("## Baseline Model Report"))
display(model_report)

display(Markdown("## Tokenizer Details"))
display(tokenizer_report)

display(Markdown("## Parameter Dtype Breakdown"))
display(dtype_breakdown(model))


## Baseline Model Report

,metric,value
0,Selected checkpoint path,sft-arithmetic-lora-demo/checkpoint-19
1,Resolved base model,Qwen/Qwen2.5-0.5B-Instruct
2,Loaded model class,Qwen2ForCausalLM
3,Total parameters,"494,032,768"
4,Trainable parameters,"494,032,768 (100.00%)"
5,Frozen parameters,0 (0.00%)
6,Shared/tied parameters counted separately by t...,"136,134,656"


## Tokenizer Details

,field,value
0,tokenizer_class,Qwen2Tokenizer
1,tokenizer_name_or_path,Qwen/Qwen2.5-0.5B-Instruct
2,pad_token,<|endoftext|>
3,eos_token,<|im_end|>
4,vocab_size,151643
5,model_max_length,131072


## Parameter Dtype Breakdown

,dtype,parameter_count
0,torch.float32,"494,032,768"
